# Examples on using the Basel Precision Instruments LNHR DAC II TelnetLib3 module

Copyright (c) Basel Precision Instruments GmbH (2025)

................................................................................................................

The LNHR DAC II provides a wide set of commands to get the most out of the device.

## 1 - imports and setup
For this example the Basel Precision Instruments LNHR DAC II Telnet driver is used (available on Github).

In [1]:
import telnet3
import asyncio

from math import sin
from math import cos
# create an instance of the LNHR DAC device
DAC = telnet3.LNHRDAC("192.168.0.5", 23) 

## 2 - set a DC voltage on a channel
The simplest form of controlling the LNHR DAC II is setting the outputs manually in a python script to a DC voltage. For best noise performance, the low bandwith should be chosen.

In [ ]:
await DAC.send_command("all off")

# set DAC channel 1 to 1 volt
await DAC.send_command("1 8CCCCC") 
await DAC.send_command("1 on")

# set multiple channels (2-6) to -2.5 volts
for channel in range(2, 5):
    await DAC.send_command(f"{channel} 600000")
    await DAC.send_command(f"{channel} lbw")
    await DAC.send_command(f"{channel} on")

[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Sent command: all off (with CRLF)
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0
0
0
[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Sent command: 1 8CCCCC (with CRLF)
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0
[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Sent command: 1 on (with CRLF)
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0
0
0
[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Sent command: 2 600000 (with CRLF)
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0
0
[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Sent command: 2 lbw (with CRLF)
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0
0
[Device-192.168.0.5] Before sending 

## 3 - update all channels simultaneously

The DAC has a synchronous update function, whith which all channels can be set to independent DC voltages at the same time. This update can be triggered by software or hardware.

In [3]:
# change update mode to synchronous
await DAC.send_command("c um-l 1")

# assign voltages to outputs
await DAC.send_command("1 80F000;2 6FFFFF;3 80F000;4 6FFFFF;5 80F000;6 6FFFFF;7 80F000;8 6FFFFF;9 80F000;10 6FFFFF;11 80F000;12 6FFFFF")

# turn channels on
await DAC.send_command("all on")

# apply voltages to outputs
await DAC.send_command("c sync-l")

[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Control command detected, waiting 0.2 seconds
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0
0
0
0
[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Sent command: 1 80F000;2 6FFFFF;3 80F000;4 6FFFFF;5 80F000;6 6FFFFF;7 80F000;8 6FFFFF;9 80F000;10 6FFFFF;11 80F000;12 6FFFFF (with CRLF)
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0;0;0;0;0;0;0;0;0;0;0;0
[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Sent command: all on (with CRLF)
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0;0;0;0;0;0;0;0;0;0;0;0
[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Control command detected, waiting 0.2 seconds
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0
0
0


## 3 - create a standard waveform with the integrated standard waveform generator
Manually setting DC voltages will neither yield nice looking waveforms nor can the results be replicated easily. 

For standard waveforms the LNHR DAC II therefore provides a set of commands to easily create well defined waveforms:
- Sine, phaseshift and DC offset applicable
- Triangular, phaseshift and DC offset applicable
- Sawtooth, phaseshift and DC offset applicable
- Ramp, phaseshift and DC offset applicable
- Rectangular, duty-cycle, phaseshift and DC offset applicable
- Gaussian Noise (fixed and random)

In [4]:
# create a simple sinewave on channel 13
await DAC.send_command("13 off")
await DAC.send_command("c awg-c stop")

while await DAC.expect_query_answer("c awg-c ava?", "0"): # wait for availability of awg-c 
    pass

await DAC.send_command("c awg-c ch 13") # select channel for awg
await DAC.send_command("c swg mode 0") # generate new waveform
await DAC.send_command("c swg wf 0") # choose waveform (0 = sine)
# further properties of waveform could be specified here, i.e. amplitude, offset or phase
await DAC.send_command("c swg wmem 2") # select wave-memory to save the generated waveform into
await DAC.send_command("c awg-c cs 0") # set number of cycles (0 = infinite cycles)
await DAC.send_command("c swg wfun 0") # generated waveform will be copied to the selected wave-memory
await DAC.send_command("c swg apply") # apply all changes to wave-memory now
await DAC.send_command("c wav-c write") # write content of wave-memory into awg memory

print("writing to memory ...", end="")
while await DAC.expect_query_answer("c wav-c busy?", "1"): # wait until waveform is written into awg memory
    print(".", end="")

await DAC.send_command("c awg-c start") # start awg
await DAC.send_command("13 on") #turn channel on

[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Sent command: 13 off (with CRLF)
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0
[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Control command detected, waiting 0.2 seconds
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0
0
[Device-192.168.0.5] Disconnected from 192.168.0.5:23
[Device-192.168.0.5] Connected to 192.168.0.5:23
[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Control command detected, waiting 0.2 seconds
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0
[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Control command detected, waiting 0.2 seconds
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0
[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Control command detected, waiti

## 4 - create a custom waveform using mathematical functions
A completely arbitrary waveform can also be defined and stored into the LNHR DAC II memory. Limiting factors are the maximum memory size of 34000 points, the maximum update rate of 100 kHz (10 us) an d the set bandwidth.

In [5]:
# create a custom waveform on channel 17
await DAC.send_command("17 off")
await DAC.send_command("c awg-d stop")

while await DAC.expect_query_answer("c awg-d ava?", "0"): # wait for availability of awg-c 
    pass

await DAC.send_command("c awg-d ch 17") # select channel for awg
await DAC.send_command("c awg-d cs 0") # set number of cycles (0 = infinite cycles)
await DAC.send_command("c wav-d clr") # clear wavememory

print("generating waveform ...", end="")
for x in range(0, 34000):
     y = 3*sin(x/200) + 2*cos(0.03*x) + (8/7)*sin(x/100) + 3.7*cos(x/10000) # generating arbitrary curve
     await DAC.send_command(f"wav-a {x:x} {y:.6f}", hold_connection=True) # transmitting datapoints as voltage to DAC
     if x % 100 == 0: print(".", end="")
print("\n")

await DAC.send_command("c wav-d write") # write content of wave-memory into awg memory

print("writing to memory ...", end="")
while await DAC.expect_query_answer("c wav-c busy?", "1"): # wait until waveform is written into awg memory
    print(".", end="")

await DAC.send_command("c awg-cd cp 1000") # set clock period, this might interfere with other awg's
await DAC.send_command("c awg-d start") # start awg
await DAC.send_command("17 on") #turn channel on

[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Sent command: 17 off (with CRLF)
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0
[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Control command detected, waiting 0.2 seconds
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0
0
0
[Device-192.168.0.5] Disconnected from 192.168.0.5:23
[Device-192.168.0.5] Connected to 192.168.0.5:23
[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Control command detected, waiting 0.2 seconds
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0
[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Control command detected, waiting 0.2 seconds
[Device-192.168.0.5] Waiting for device response...
[Device-192.168.0.5] Device response: 0
[Device-192.168.0.5] Before sending command
[Device-192.168.0.5] Control command detected, wai